In [2]:
from utils.spark import build_spark_explore

spark = build_spark_explore()
spark.sql("SHOW TABLES").show()


+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  default|         hrrr_bronze|      false|
|  default|hrrr_gold_capital...|      false|
|  default|hrrr_gold_capital...|      false|
|  default|         hrrr_silver|      false|
|  default|         us_capitals|      false|
+---------+--------------------+-----------+



In [4]:
df = spark.read.table("hrrr_bronze")
# df = spark.sql("SELECT * FROM hrrr_bronze")
print("Table Schema:")
df.printSchema()


Table Schema:
root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [ ]:
df.select("path", "modificationTime", "length", "ingested_at").show(5, truncate=False)


Sample Data:


+---------------------------------------------------------------------+-------------------+---------+-------------------------+
|path                                                                 |modificationTime   |length   |ingested_at              |
+---------------------------------------------------------------------+-------------------+---------+-------------------------+
|s3a://noaa-hrrr-bdp-pds/hrrr.20260829/conus/hrrr.t22z.wrfsfcf00.grib2|2026-08-29 17:03:10|153101608|2026-09-24 11:09:53.50405|
|s3a://noaa-hrrr-bdp-pds/hrrr.20260921/conus/hrrr.t15z.wrfsfcf00.grib2|2026-09-21 09:51:18|151001739|2026-09-24 11:09:53.50405|
|s3a://noaa-hrrr-bdp-pds/hrrr.20260920/conus/hrrr.t23z.wrfsfcf00.grib2|2026-09-20 17:52:13|149512469|2026-09-24 11:09:53.50405|
|s3a://noaa-hrrr-bdp-pds/hrrr.20260831/conus/hrrr.t00z.wrfsfcf00.grib2|2026-08-30 18:52:48|146910481|2026-09-24 11:09:53.50405|
|s3a://noaa-hrrr-bdp-pds/hrrr.20260825/conus/hrrr.t16z.wrfsfcf00.grib2|2026-08-25 10:52:11|146736851|202

In [24]:
from pyspark.sql.functions import split, element_at


(
    df.select("ingested_at", "modificationTime")
    .withColumn("modificationDate", element_at(split("modificationTime", " "), 1))
    .drop("modificationTime")
    .orderBy("modificationDate", "ingested_at", ascending=False).show(30, truncate=False)
)

+-----------------------+----------------+
|ingested_at            |modificationDate|
+-----------------------+----------------+
|2026-09-29 08:05:12.956|2026-09-29      |
|2026-09-29 08:05:12.956|2026-09-29      |
|2026-09-29 08:05:12.956|2026-09-29      |
|2026-09-29 08:05:12.956|2026-09-29      |
|2026-09-29 08:05:12.956|2026-09-29      |
|2026-09-29 08:05:12.956|2026-09-29      |
|2026-09-29 08:05:12.956|2026-09-29      |
|2026-09-29 08:05:12.956|2026-09-29      |
|2026-09-29 08:05:12.956|2026-09-28      |
|2026-09-29 08:05:12.956|2026-09-28      |
|2026-09-29 08:05:12.956|2026-09-28      |
|2026-09-29 08:05:12.956|2026-09-28      |
|2026-09-29 08:05:12.956|2026-09-28      |
|2026-09-29 08:05:12.956|2026-09-28      |
|2026-09-28 18:05:10.263|2026-09-28      |
|2026-09-28 17:05:11.645|2026-09-28      |
|2026-09-28 16:05:10.901|2026-09-28      |
|2026-09-28 15:05:11.06 |2026-09-28      |
|2026-09-28 14:05:11.343|2026-09-28      |
|2026-09-28 13:05:11.686|2026-09-28      |
|2026-09-28

In [14]:
import tempfile
import os
import pandas as pd
from pyspark.sql import functions as F
from utils.constants import BRONZE_TABLE_PATH
from pipelines.bronze_to_silver import _decode_file


bronze_row = (
    spark.read.format("delta")
    .load(BRONZE_TABLE_PATH)
    .limit(1)
    .select("path", "content")
    .collect()[0]
)

source_path = bronze_row["path"]
raw_bytes = bronze_row["content"]

print(f"Processing bronze file: {source_path}")

# 2. Write raw bytes to a temp file and decode using your pipeline's function
with tempfile.NamedTemporaryFile(delete=False, suffix=".grib2") as tmp:
    tmp.write(raw_bytes)
    tmp_path = tmp.name

try:
    df_decoded = _decode_file(tmp_path)
    df_decoded["source_path"] = source_path
    df_decoded["decode_error"] = None
    print(f"Successfully decoded {len(df_decoded):,} grid points!")
except Exception as exc:
    print(f"Failed to decode: {exc}")
    df_decoded = pd.DataFrame()
finally:
    if os.path.exists(tmp_path):
        os.remove(tmp_path)

display(df_decoded.head(20))

Processing bronze file: s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hrrr.t22z.wrfsfcf00.grib2
Successfully decoded 1,905,141 grid points!


,valid_time,latitude,longitude,sp,gust,vis,prate,crain,csnow,cape,...,sulwrf,ishf,slhtf,gflux,t_2m,dpt,u,v,source_path,decode_error
0,2026-09-28 22:00:00,21.138123,237.280472,100690.0,10.260437,13600.0,0.0,0.0,0.0,100.0,...,441.374756,2.0,70.0,0.0,298.376282,296.810699,8.23641,-2.369267,s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hr...,None
1,2026-09-28 22:00:00,21.145109,237.307144,100690.0,10.260437,13600.0,0.0,0.0,0.0,110.0,...,441.374756,2.0,70.0,0.0,298.376282,296.810699,8.17391,-2.244267,s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hr...,None
2,2026-09-28 22:00:00,21.152090,237.333817,100690.0,10.197937,13600.0,0.0,0.0,0.0,120.0,...,441.249756,2.0,69.0,0.0,298.376282,296.810699,8.17391,-2.056767,s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hr...,None
3,2026-09-28 22:00:00,21.159063,237.360489,100690.0,10.135437,13600.0,0.0,0.0,0.0,120.0,...,441.249756,1.0,68.0,0.0,298.376282,296.810699,8.17391,-1.931767,s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hr...,None
4,2026-09-28 22:00:00,21.166029,237.387161,100690.0,10.072937,13600.0,0.0,0.0,0.0,120.0,...,441.124756,1.0,67.0,0.0,298.376282,296.810699,8.11141,-1.744267,s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hr...,None
5,2026-09-28 22:00:00,21.172989,237.413849,100690.0,10.072937,13600.0,0.0,0.0,0.0,120.0,...,440.999756,1.0,65.0,0.0,298.376282,296.810699,8.11141,-1.619267,s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hr...,None
6,2026-09-28 22:00:00,21.179941,237.440536,100690.0,10.010437,13600.0,0.0,0.0,0.0,120.0,...,440.874756,1.0,64.0,0.0,298.376282,296.810699,8.11141,-1.431767,s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hr...,None
7,2026-09-28 22:00:00,21.186886,237.467224,100690.0,9.947937,13500.0,0.0,0.0,0.0,120.0,...,440.749756,1.0,63.0,0.0,298.376282,296.810699,8.04891,-1.306767,s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hr...,None
8,2026-09-28 22:00:00,21.193825,237.493927,100690.0,9.947937,13500.0,0.0,0.0,0.0,120.0,...,440.624756,1.0,61.0,0.0,298.313782,296.810699,8.04891,-1.119267,s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hr...,None
9,2026-09-28 22:00:00,21.200756,237.520630,100690.0,9.885437,13500.0,0.0,0.0,0.0,120.0,...,440.499756,1.0,60.0,0.0,298.313782,296.810699,7.98641,-0.994267,s3a://noaa-hrrr-bdp-pds/hrrr.20260928/conus/hr...,None
